<img src="logo.png" alt="Vegeta" width="240">

# Project Sikarian — Sikarian Lobster Nefri, the small underwater walker

The **Sikarian Lobster** is the Sikarian line's underwater member (after the robot dog and the Myropods): a
reconnaissance and blocking node for river crossings, bridge piers, harbours and outfall pipes. It **walks on the
bottom** on six simple legs, **cuts and grips** with two pincers, and **swims** on one thruster — a propeller in a
shroud at the end of a **jointed tail** that points the jet wherever it is needed: forward and up to swim, sideways to
turn, down onto the back to press the robot onto the bottom in a current.

**Nefri** is v1, the small member (*Nephrops*, the Norway lobster): ~7 kg, 0.32 m shell, 0–8 m of fresh water. Its
job (inputs, from the brief): **cut a Ø10 mm polypropylene rope** strung across the mouth of a flooded **Ø600 mm
outfall pipe** and **walk into the pipe**. The big member, **Ornatus** (15 kg, a Ø12 mm electric cable), is the next
notebook. This one checks Nefri end to end:

```
CAD (Dedalus): shell, pressure housing, six 2-joint legs, two 4-joint pincer arms, a 2-segment tail with a shroud
    and a 76 mm propeller ─► masses, displaced volumes, the foam that trims it to sink with 10 % of its weight,
    the centre of buoyancy over the centre of gravity
standing and walking: leg loads in water (and why it lies on its belly out of it), the servos, the tripod gait
the thruster: Boreas blade element theory in water, the duct, cavitation with depth, the swim speed
the tail: where it can point the thrust — the line through the centre of gravity, the curl over the back
CFD (Aeromant, OpenFOAM, fast presets): the propeller at bollard vs BEMT; the whole body with the jet deflected
the pincer: the cutting force against rope classes, the jaw under the worm drive's stall (FEA)
FEA (Talos): the pressure housing at 8 m (+ buckling by hand), a leg, a tail segment; tail modes vs blade pass
endurance: standby, watch, walk, swim
MuJoCo (ChironLab, water): swim and land; cut the rope and walk into the pipe; hold in a 0.5 m/s current — movies
```

*Long cells: the CFD (§5, minutes each on the fast preset; `VEGETA_SKIP_OPENFOAM=1` skips them) and the three
missions with their movies (§9, ~10 min). Everything else runs in a minute or two.*

In [ ]:
import json, math, os, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video
from vegeta import dedalus, talos, chiron, aeromant, boreas
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.chiron import viz as cviz
from vegeta.aeromant import viz as aviz
from vegeta.aeromant import movie as amovie
sys.path.insert(0, "designs")
import actuators as act
import lobster, lobster_robot as lr, lobster_controller as lc, lobster_scenario as ls, lobster_cfd as cfd

RUNS = Path("_runs/sikarian_lobster_nefri"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
OUT = Path("output/24_sikarian_lobster_nefri"); OUT.mkdir(parents=True, exist_ok=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM", "0") != "1"
G, RHO = lr.WATER["g"], lr.WATER["density"]
design = lobster.SikarianLobster()
p = lr.design_params()
JOB = pd.Series({"name": "Sikarian Lobster Nefri (v1, small)", "water": "fresh, 0-8 m, 1000 kg/m³", "mass class [kg]": "~6-7",
                 "the rope": f"Ø{ls.ROPE['diameter_m'] * 1000:.0f} mm PP, cut at {ls.F_CUT:.0f} N", "the pipe": f"Ø{ls.PIPE['inner_d'] * 1000:.0f} mm inner, {ls.PIPE['silt'] * 1000:.0f} mm of silt in it",
                 "current to hold in [m/s]": 0.5, "standby [h]": "12-48"}, name="Nefri — the job (inputs)")
display(JOB.to_frame())
pd.DataFrame(design.params.table()).set_index("name").loc[[k for k in lr.NEFRI]].iloc[:, :3]

## 1. The machine: CAD, masses, volumes, trim

A faceted ASA shell (free-flooding: drain slots in the belly, water inside) over a sealed **Al 6082 pressure housing**
(Ø110 × 230 mm, O-ring end caps: the battery, the computer, the IMU and the depth sensor), syntactic foam in the
shell's upper corners. Six PA12-CF legs on sealed 1.5 N·m servos (hip yaw swings the leg, hip pitch lifts it), two
pincer arms (shoulder yaw, shoulder, elbow, wrist; a jaw pair on a pin driven by a self-locking worm servo: the right
jaw has the cutter notch near its pin), cameras and a sonar on the nose, masts on the roof. The tail: two Ø50 mm
segments, each on a yaw and a pitch servo, ending in a shroud (Ø82 inner) with the 76 mm three-blade propeller.

**Buoyancy.** Every part has a mass and a displaced volume (the CAD's volumes × densities, the catalogue's servos,
listed parts); the housing displaces its whole envelope. The foam volume is solved so the robot sinks with
`NET_FRACTION` = 10 % of its weight — heavy enough to stand and walk, light enough for a small thrust to lift it.
**Trim**: the foam sits high (35 mm over the housing axis) and the battery low (35 mm under it), so the robot rights
itself in the water; the foam's position along x is solved so the centre of buoyancy is right over the centre of
gravity when it stands (`lr.lobster(trim=True)`) — otherwise it would swim nose-down.

In [ ]:
parts = {k: design.generate(**dict(p, part=k)) for k in ("robot", "shell", "housing", "leg", "upper_arm", "jaw", "tail_segment", "shroud", "propeller")}
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.1 if k in ("jaw", "propeller") else 0.3) for k, g in parts.items()}
cad_num = lr.cad_numbers(p, recompute=True)
assert max(abs(cad_num[k] / lr.CAD[k] - 1) for k in lr.CAD) < 0.01, "lobster_robot.CAD is stale: update it from cad_num"
dviz.show(dviz.plot3d(parts["robot"]))

In [ ]:
b = lr.budget(p, cad_num)
mass = pd.DataFrame({"mass [kg]": {k: v[0] for k, v in b["parts"].items()}, "volume [L]": {k: v[1] * 1000 for k, v in b["parts"].items()}})
mass["buoyancy − weight [N]"] = (RHO * mass["volume [L]"] / 1000 - mass["mass [kg]"]) * G
mass.loc["TOTAL"] = mass.sum()
robot = lr.lobster(cad=cad_num)
print(f"Nefri {b['mass']:.2f} kg, displaces {b['volume'] * 1000:.2f} L: in fresh water it weighs {b['wet_weight_N']:.2f} N ({b['net_kg'] / b['mass'] * 100:.0f} % of its weight); "
      f"in sea water (1025) {(b['mass'] - 1025 * b['volume']) * G:.2f} N")
print(f"foam {b['foam_volume'] * 1000:.2f} L in {b['foam_space'] * 1000:.2f} L of room beside the housing | trim: foam centre at x = {robot.trim['foam_x_m'] * 1000:+.0f} mm "
      f"(the pitching couple {robot.trim['couple_untrimmed_Nm']:+.3f} N·m untrimmed, {robot.trim['couple_trimmed_Nm']:+.4f} N·m trimmed)")
mass.round(3)

In [ ]:
lab_flat = lr.lobster_lab(chiron.Flat(), robot=robot)
ep_stand = lab_flat.run(lc.Mission([lc.Phase("stand", lc.wait(3.0), None, 4.0)], name="stand"), duration=3.0, rules=None, settle=0.0)
fz = np.asarray(ep_stand.log["foot_force"])[-1, :, 2]
bq = np.asarray(ep_stand.log["body_quat"])[-1, 0]
print(f"standing in water (MuJoCo, 3 s): feet carry {fz.sum():.2f} N (the wet weight {b['wet_weight_N']:.2f} N), per leg {fz.round(2)} N; "
      f"pitch {math.degrees(lc.pitch_of(bq)):+.2f}°, hull centre {np.asarray(ep_stand.log['body_pos'])[-1, 0, 2] * 1000:.0f} mm over the bed")

## 2. Standing and walking on the bottom

**Leg loads.** On a tripod (three feet down) each foot carries a third of the wet weight; the hip's lift servo holds
it at the foot's reach (femur + the splayed tibia). The sizing case is not standing but **pressing**: in a current
the tail pushes the robot down with up to ~21 N (§4, §9) and the legs carry that too. **Out of the water** — on a
deck, at the launch — the full weight on a tripod would need more than the servos have: Nefri lies on its belly there
(it is carried, not walked, on land).

**Current.** The drag of a cross-current on the shell's side (`½ ρ C_d A v²`, C_d 1.0 for the faceted box with its
legs) against the friction its wet weight gives (μ 0.6 on silt and gravel, an input): the current it holds in
standing, and the downward thrust that holds it at 0.5 m/s.

**Gait.** The tripod gait (FL·MR·RL against FR·ML·RR, duty 0.5, 1.2 s cycle): the hip yaw swings the planted legs
back, the lift raises the others and swings them forward. The stride shrinks with the commanded speed (short steps to
creep onto the rope). A short walk in MuJoCo gives the speed.

In [ ]:
g = lr.geometry(p)
reach = g["foot"][1]                                                # hip to foot, horizontal [m]
LEG = act.get(lr.SERVO_KEYS["leg"])
mu_bed, cd_side = 0.6, 1.0                                          # inputs
A_side = g["shell"][0] * g["shell"][2] * 1.3                        # the shell's side + legs and arms (×1.3)
W_wet, W_air = b["wet_weight_N"], b["mass"] * G
T_press = lr.thrust(lr.thruster_table(p), 2400, 0.0)[0]            # the press of §9 (2400 rpm, the tail curled)
cases = {"standing in water, tripod": W_wet / 3, "pressing with 2400 rpm of thrust, tripod": (W_wet + T_press) / 3,
         "in air on six legs": W_air / 6, "in air on a tripod": W_air / 3}
legs = pd.DataFrame({"foot load [N]": cases})
legs["hip lift torque [N·m]"] = legs["foot load [N]"] * reach
legs["servo stall / torque"] = LEG.stall_Nm / legs["hip lift torque [N·m]"]
legs["servo rated / torque"] = LEG.rated_Nm / legs["hip lift torque [N·m]"]
v_slide = math.sqrt(2 * mu_bed * W_wet / (RHO * cd_side * A_side))
F_cur = 0.5 * RHO * cd_side * A_side * 0.5 ** 2
T_hold = F_cur / mu_bed - W_wet                                    # extra normal force the thrust must add at 0.5 m/s
print(f"leg servo: {LEG.key} (stall {LEG.stall_Nm} N·m, rated {LEG.rated_Nm} N·m); foot reach {reach * 1000:.0f} mm")
print(f"current: side area {A_side:.3f} m² → standing it slides above {v_slide:.2f} m/s; at 0.5 m/s the drag is {F_cur:.1f} N and the thrust must add "
      f"{T_hold:.1f} N of normal force (the press gives {T_press:.1f} N at 2400 rpm)")
legs.round(2)

In [ ]:
lab_walk = lr.lobster_lab(chiron.Flat(), robot=robot)
walk = lc.Mission([lc.Phase("stand", lc.wait(0.5), None, 1.0), lc.Phase("walk", lc.walk_to(0.6, v=0.10), None, 15.0),
                   lc.Phase("stand", lc.wait(0.6), None, 1.0)], name="walk 0.6 m")
ep_walk = lab_walk.run(walk, duration=10.0, rules=None, settle=0.0)
tw = ls.timeseries(ep_walk)
st = {r[1]: r[0] for r in walk.log if r[2] == "start"}; dn = [r[0] for r in walk.log if r[1] == "walk" and r[2] == "done"]
t0, t1 = st["walk"], (dn[0] if dn else tw.t.iloc[-1])
sel = tw[(tw.t > t0 + 1.2) & (tw.t < t1)]
v_walk = (sel.x.iloc[-1] - sel.x.iloc[0]) / (sel.t.iloc[-1] - sel.t.iloc[0])
names = list(ep_walk.log["joints"]); tau = np.abs(np.asarray(ep_walk.log["tau"]))
lift_peak = max(tau[:, names.index(f"{leg}_lift")].max() for leg in lr.LEGS)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].plot(tw.t, tw.x, label="hull x [m]"); ax[0].plot(tw.t, tw.tilt_deg / 10, label="tilt / 10 [deg]"); ax[0].plot(tw.t, tw.foot_load_N / 10, label="feet / 10 [N]")
ax[0].set(xlabel="time [s]", title=f"tripod walk: {v_walk:.3f} m/s (commanded 0.10)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for leg in lr.LEGS:
    ax[1].plot(ep_walk.log["t"], tau[:, names.index(f"{leg}_lift")], lw=0.8, label=leg)
ax[1].axhline(LEG.rated_Nm, color="k", ls="--", lw=0.8); ax[1].set(xlabel="time [s]", ylabel="|τ| hip lift [N·m]", title="hip lift servos (dashed: rated)"); ax[1].legend(fontsize=7, ncol=3); ax[1].grid(alpha=0.3)
fig.tight_layout()
print(f"walking {v_walk:.3f} m/s; peak hip lift torque {lift_peak:.2f} N·m (rated {LEG.rated_Nm}, stall {LEG.stall_Nm})")

## 3. The thruster: blade element theory in water, the duct, cavitation, the swim

The 76 mm three-blade propeller (pitch 70 mm) in **Boreas** (blade element momentum theory, the marine section of
notebooks 13/14) in water at 0–3300 rpm and 0–1.5 m/s of inflow: thrust, torque, the 250 kV motor's current and
efficiency on the 4S pack. **The duct**: a shroud that recovers the jet's contraction raises the bollard thrust at
the same power — momentum theory gives (2σ)^⅓ = 1.34 for an exit-to-disc ratio σ = 1.2 (`lobster_cfd.duct_gain`);
the MuJoCo thruster takes **1.2** (`lr.DUCT_GAIN`) for the short, thick-lipped duct and its strut.
**Cavitation**: the cavitation number at 0.7 R against a generic section's −C_p,min = 1 (`boreas.cavitation`) at the
surface and at 8 m. **The swim**: the thrust at speed against the hull's drag gives the top speed; the drag of the
shell with its tucked legs and arms is an input here (C_d·A from MuJoCo's own fluid model, which the §9 swim uses).

In [ ]:
tab = lr.thruster_table(p, rpm=np.linspace(0, lr.THRUSTER["rpm_max"], 23), speed=np.array([0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]))
mot, bat = lr.THRUSTER["motor"], lr.THRUSTER["battery"]
FW = boreas.noise.Medium("fresh water", RHO, 1480.0)
prop = lr._prop(p)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for j, v in enumerate(tab["speed"]):
    ax[0].plot(tab["rpm"], tab["thrust"][:, j] * lr.DUCT_GAIN, label=f"{v:.2f} m/s")
ax[0].set(xlabel="rpm", ylabel="thrust (ducted) [N]", title="thrust × DUCT_GAIN vs rpm and inflow"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
rows = {}
for rpm in (1500, 2000, 2400, 2600, 3000, 3300):
    T, Q = lr.thrust(tab, rpm, 0.0)
    V, I = mot.voltage_for(rpm, Q)
    rows[rpm] = {"thrust (ducted) [N]": T, "torque [N·m]": Q, "shaft [W]": Q * rpm * 2 * math.pi / 60, "motor [V]": V, "motor [A]": I,
                 "electric [W]": V * I, "motor η": mot.efficiency(rpm, Q), "fits the pack": V <= bat.voltage * 1.02 and I <= mot.max_current_a}
bollard = pd.DataFrame(rows).T
depths = np.linspace(0, 8, 33)
for rpm in (2400, 3300):
    ax[1].plot(depths, [boreas.cavitation(prop, rpm, 0.0, d, FW)["cavitation_number"] for d in depths], label=f"{rpm} rpm")
ax[1].axhline(1.0, color="r", ls="--", lw=0.8, label="−Cp,min = 1: cavitation below")
ax[1].set(xlabel="depth [m]", ylabel="σ at 0.7 R", title="cavitation number at bollard"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
# the swim: thrust at speed vs drag; C_d·A of the swimming body (an input: what MuJoCo's ellipsoid model gives, see §9)
CDA = 0.069                                                          # m²
vs = np.linspace(0, 1.5, 61)
for rpm in (2000, 2600, 3300):
    ax[2].plot(vs, [lr.thrust(tab, rpm, v)[0] for v in vs], label=f"thrust {rpm} rpm")
ax[2].plot(vs, 0.5 * RHO * CDA * vs ** 2, "k", label=f"drag (C_d·A {CDA} m²)")
ax[2].set(xlabel="speed [m/s]", ylabel="N", title="the swim: thrust = drag", ylim=(0, 45)); ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)
fig.tight_layout()
v_swim = {rpm: round(float(vs[np.argmin(np.abs([lr.thrust(tab, rpm, v)[0] - 0.5 * RHO * CDA * v ** 2 for v in vs]))]), 2) for rpm in (2000, 2600, 3300)}
cav = pd.DataFrame({f"{d:g} m": {rpm: boreas.cavitation(prop, rpm, 0.0, d, FW)["cavitation_number"] for rpm in (2400, 2600, 3300)} for d in (0, 1, 8)})
print(f"duct gain by momentum theory {cfd.duct_gain():.2f} (σ 1.2), taken {lr.DUCT_GAIN} | swim speed {v_swim} m/s | blade pass at 3300 rpm {3 * 3300 / 60:.0f} Hz")
display(cav.round(2).rename_axis("σ at rpm"))
bollard.round(3)

## 4. The tail: where it can point the thrust

The tail's two pitch joints set the thrust's angle (θ = −(q₁ + q₂) above the hull's x) **and** where its line passes:
`thrust_line(θ)` solves q₁, q₂ so the line goes through the centre of gravity — a pure force, no pitching moment.
The tail is short: through the CG it can point the thrust only within about ±15° of the hull's axis — enough to swim
level, climb and dive gently (the swim of §9 holds its depth with this and steers with the yaw joints). Steeper
thrust **pitches** the robot: steep *down* thrust (the jet up, pressing the robot onto the bottom) is done with the
tail **curled over the back** and the line through a point between the feet (`cg=(−0.08, 0, 0)`): the moment it
leaves is taken by the legs on the bottom. The tail servos see the thrust's moment about joint 1 (`T · L₁ · sin q₂`)
and the propeller's torque.

In [ ]:
thetas = np.arange(-90, 91, 5)
TAIL = act.get(lr.SERVO_KEYS["tail"])
env_rows = {}
for th in thetas:
    q = lc.thrust_line(math.radians(th))
    qb = lc.thrust_line(math.radians(th), best_effort=True)
    qc = lc.thrust_line(math.radians(th), cg=(-0.08, 0.0, 0.0))
    row = {"through the CG": q is not None, "between the feet": qc is not None}
    for name, qq in (("CG", q), ("feet", qc)):
        if qq is not None:
            row[f"q1 {name} [deg]"], row[f"q2 {name} [deg]"] = math.degrees(qq[0]), math.degrees(qq[1])
    env_rows[th] = row
envelope = pd.DataFrame(env_rows).T
T26 = lr.thrust(tab, 2600, 0.0)[0]; T24 = lr.thrust(tab, 2400, 0.0)[0]; Tmax = lr.thrust(tab, lr.THRUSTER["rpm_max"], 0.0)[0]
def joint1(q2, T):
    return T * g["L1"] * abs(math.sin(q2))
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ok = envelope[envelope["through the CG"] == True]
okc = envelope[envelope["between the feet"] == True]
ax[0].plot(ok.index, ok["q1 CG [deg]"], "o-", label="q1 (through the CG)"); ax[0].plot(ok.index, ok["q2 CG [deg]"], "o-", label="q2 (through the CG)")
ax[0].plot(okc.index, okc["q1 feet [deg]"], "s--", label="q1 (between the feet)"); ax[0].plot(okc.index, okc["q2 feet [deg]"], "s--", label="q2 (between the feet)")
ax[0].set(xlabel="thrust angle θ above the hull's x [deg]", ylabel="tail pitch [deg]", title="the tail's thrust-line envelope"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for T, lab_ in ((T24, "2400 rpm"), (T26, "2600 rpm"), (Tmax, f"{lr.THRUSTER['rpm_max']:.0f} rpm")):
    ax[1].plot(okc.index, [joint1(math.radians(v), T) for v in okc["q2 feet [deg]"]], label=f"{lab_}, between the feet")
ax[1].axhline(TAIL.stall_Nm, color="r", ls="--", lw=0.8, label="tail servo stall"); ax[1].axhline(TAIL.rated_Nm, color="k", ls=":", lw=0.8, label="rated")
ax[1].set(xlabel="θ [deg]", ylabel="joint 1 [N·m]", title="the thrust's moment at tail joint 1 (bollard)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()
q_press = lc.thrust_line(math.radians(-60), cg=(-0.08, 0, 0))
print(f"through the CG: θ from {ok.index.min()}° to {ok.index.max()}° | between the feet: {okc.index.min()}° to {okc.index.max()}° | "
      f"the press (θ −60°, 2400 rpm): q = {np.degrees(q_press).round(0)}°, joint 1 {joint1(q_press[1], T24):.2f} N·m (servo stall {TAIL.stall_Nm}) | "
      f"propeller torque {lr.thrust(tab, 2600, 0)[1]:.2f} N·m at 2600 rpm")

## 5. CFD (Aeromant, OpenFOAM): the propeller in water, and the vectored jet on the body

**The propeller at bollard** (`rotor_mrf_static`: the blades resolved in a rotating frame, steady k-ω SST, a small
residual inflow for a stable steady solution) against Boreas: thrust, torque, the figure of merit, and tracer
particles through the disc (`aeromant.movie.make_movie`). The shroud is not in this case (an MRF zone cannot hold a
duct still around a turning propeller): its gain is §3's.

**The vectored jet** (`hull_rotor_disk`): the whole body swimming at 0.6 m/s (the `cfd` part: shell, tail segments and
shroud, posed; legs and arms left out), the propeller as a **rotor disk** — blade-element source terms from the same
Boreas blade and section — along the shroud axis, the tail straight and yawed to deflect the jet 35°. At the swim
the propeller is close to bollard (the jet ~5× the stream) and the rotor disk with the local cell velocity diverges
within a few iterations; the case uses the template's `fixed_inflow` (the blade elements see the free stream: a
steady source) at a **calibrated rpm** — the one whose no-induction thrust equals Boreas' open-propeller thrust at
2600 rpm and the swim (`lobster_cfd.disk_calibration`). The solver log's disk thrust is in the table next to it.
The disk's thrust is not in the body's forces; what the case adds to the thrust vector is the jet **scrubbing the
shell** and the shroud's own lift: the side force and the yaw moment on the body against the simple picture
(`T sin δ`). The CFD frame has the stream along +x, so the body faces −x (`lobster_cfd.to_robot` turns vectors back).

*Fast presets (`lobster_cfd.QUALITY`): the body case meshes to ~50 k cells in half a minute and solves 300
iterations in a few minutes; `quality="fine"` for the numbers you'd quote. Forces within ~20-30 % on the fast mesh.*

In [ ]:
env = None
try:
    env = aeromant.OpenFOAMEnvironment.detect()
    print("OpenFOAM", env.version(), env.flavor())
except Exception as exc:
    print("no OpenFOAM:", exc); RUN_CFD = False
CFD_QUALITY = "fast"
prop_stl = cfd.prop_stl(RUNS / "cad")
case_prop = cfd.prop_case(prop_stl, RUNS / "cfd_prop", p, quality=CFD_QUALITY, environment=env)
print(case_prop.prepare(overwrite=True))
aviz.show(aviz.plot_setup(case_prop))

In [ ]:
bemt = cfd.bemt_bollard(p=p)
res_prop = None
if RUN_CFD:
    res_prop = case_prop.run(processors=4, progress=True)
    print(res_prop.ok, res_prop.messages[-2:])
comp = {"BEMT (Boreas), open": bemt}
if res_prop is not None and res_prop.ok:
    mp = res_prop.metrics
    comp["CFD (rotor_mrf_static), open"] = {"rpm": bemt["rpm"], "thrust_N": mp["thrust_N"], "torque_Nm": mp["torque_Nm"], "power_W": mp["power_W"]}
    print(f"{mp.get('mesh_cells')} cells, converged {mp.get('converged')} in {mp.get('iterations')} iterations; figure of merit {mp.get('figure_of_merit', float('nan')):.2f}")
compare_prop = pd.DataFrame(comp).T
if len(comp) > 1:
    compare_prop.loc["CFD / BEMT"] = compare_prop.iloc[1] / compare_prop.iloc[0]
display(compare_prop.round(3))
if res_prop is not None and res_prop.ok:
    aviz.show(aviz.plot_field_slice(case_prop, "U", normal="z"))
    clip = amovie.make_movie(case_prop, OUT / "lobster_propeller_bollard.mp4", blades=int(p["prop_blades"]), n=40, seconds=8, fps=24, degrees_per_frame=10,
                             title=f"Nefri's 76 mm propeller at bollard, {bemt['rpm']:.0f} rpm in water (rotor_mrf_static): tracer particles")
    display(Video(str(clip), embed=False, width=900))

In [ ]:
DEFLECTIONS = (0.0, 35.0)
tail_cases, tail_res = {}, {}
for d in DEFLECTIONS:
    stl = cfd.body_stl(RUNS / "cad", d, p)
    tail_cases[d] = cfd.tail_case(stl, RUNS / f"cfd_tail_{d:g}", d, p, quality=CFD_QUALITY, environment=env)
    pr = tail_cases[d].prepare(overwrite=True)
    cal = cfd.disk_calibration(d, p)
    print(f"jet deflected {d:g}°: {pr.ok}, disk at {np.round(tail_cases[d].parameters['disk1_center'], 3).tolist()} m, axis {np.round(tail_cases[d].parameters['disk_axis'], 3).tolist()}; "
          f"BEMT {cal['target_thrust_N']:.1f} N at {cal['rpm']:.0f} rpm → the disk at {cal['disk_rpm']:.0f} rpm (no induction: {cal['uncalibrated_thrust_N']:.1f} N at {cal['rpm']:.0f})")
aviz.show(aviz.plot_setup(tail_cases[35.0]))

In [ ]:
rows = {}
if RUN_CFD:
    for d, case in tail_cases.items():
        tail_res[d] = case.run(processors=4, progress=True)
        print(d, tail_res[d].ok, tail_res[d].messages[-2:])
        if tail_res[d].ok:
            f = cfd.tail_forces(case)
            T_swim = cfd.disk_calibration(d, p)["target_thrust_N"]
            simple = cfd.jet_side_force(d, T_swim)
            rows[f"{d:g}°"] = {"body drag [N]": f["drag_N"], "body side force, robot y [N]": f["force_robot_N"][1], "body lift [N]": f["lift_N"],
                               "body yaw moment [N·m]": f["yaw_moment_Nm"], "disk thrust (solver log) [N]": f.get("effective_lift", float("nan")),
                               "BEMT thrust, open [N]": T_swim, "T sin δ [N]": simple["side_N"], "cells": tail_res[d].metrics.get("mesh_cells")}
jet = pd.DataFrame(rows).T
display(jet.round(3))
if 35.0 in tail_res and tail_res[35.0].ok:
    aviz.show(aviz.plot_field_slice(tail_cases[35.0], "U", normal="z", origin=(0, 0, float(tail_cases[35.0].parameters["disk1_center"][2]))))
    clip = aviz.animate_particles(tail_cases[35.0], OUT / "lobster_tail_jet_35deg.mp4", n=400, seconds=8, fps=24, lengths=4.0, progress=True)
    display(Video(str(clip), embed=False, width=900))

## 6. The pincer: the cutting force, rope classes, the jaw under load

The right pincer's cutter is a hardened V notch 14 mm from the jaw pin; the jaws close by one **self-locking worm
servo** (12 N·m stall, 5 N·m rated; the mechanism's linkage sends its torque to both jaws). At the notch the squeeze
is `τ / 14 mm`: 860 N at stall, 360 N continuous. The rope classes below are inputs (blade-against-anvil cutting
forces of the class; a cutter test on the actual rope sets them). In MuJoCo the rope parts when **both** jaws press
it with `F_CUT` = 600 N for 50 ms (the Onager Manus's `WireCutter` hook): the contact forces decide. The notch's outer
flank is a 4 mm tooth (`lr.NOTCH_TOOTH`): without it, closing jaws push a round rope outward along the blade
(scissors) to where the squeeze is only ~340 N.

**Jaw FEA** (tool steel, its own frame): the pin bore held, the stall squeeze on the notch.

In [ ]:
JAW = act.get(lr.SERVO_KEYS["jaw"])
F_stall, F_rated = JAW.stall_Nm / g["cutter"], JAW.rated_Nm / g["cutter"]
ROPES = {"PP 3-strand Ø6": 250.0, "PP 3-strand Ø10 (the job)": ls.F_CUT, "PA (nylon) Ø10": 900.0, "polyester Ø10": 1000.0,
         "HMPE (Dyneema) Ø6": 1500.0, "PVC cable 3×2.5 mm², Ø12 (Ornatus' job)": 2500.0}          # inputs: cutting forces of the class [N]
ropes = pd.DataFrame({"cutting force [N]": ROPES})
ropes["stall squeeze / force"] = F_stall / ropes["cutting force [N]"]
ropes["Nefri cuts it"] = ropes["stall squeeze / force"] >= 1.0
print(f"squeeze at the notch: {F_stall:.0f} N at stall, {F_rated:.0f} N rated (worm {JAW.key}, self-locking: it holds a grip unpowered)")
display(ropes.round(2))
TOOL = talos.Material("tool steel jaw (42CrMo4 QT)", youngs_modulus=210000.0, poissons_ratio=0.30, density=7.85e-9, yield_strength=900.0, source="EN 10083-3")
jt, jd, nx_, nd_ = p["jaw_thickness"], p["jaw_depth"], p["cutter_x"], p["cutter_depth"]
rp = p["jaw_pin_diameter"] / 2 + 0.5
JAW_REGIONS = [talos.SurfacesInBox("pin", (-rp, -jt, jd / 2 - rp, rp, jt, jd / 2 + rp)),
               talos.SurfacesInBox("notch", (nx_ - nd_ - 0.5, -jt, -1.5, nx_ + nd_ + 0.5, jt, nd_ + 0.5))]
jm = talos.StructuralModel(cad["jaw"].artifacts["step"], "mm-N-MPa", TOOL, JAW_REGIONS, [talos.FixedSupport("pin")], [talos.Force("notch", fz=F_stall)],
                           talos.MeshSettings(element_size=1.5, min_element_size=0.4), name="jaw_cut")
jm.mesh(RUNS / "jaw_cut"); jaw_res = jm.solve(RUNS / "jaw_cut")
print(f"jaw at {F_stall:.0f} N on the notch: max von Mises {jaw_res.metrics.get('max_von_mises'):.0f} MPa, SF {jaw_res.metrics.get('safety_factor_yield'):.2f}, "
      f"max displacement {jaw_res.metrics.get('max_displacement'):.3f} mm")
tviz.show(tviz.plot_results(jaw_res, field="von_mises"))

## 7. FEA (Talos): the pressure housing at 8 m, a leg, a tail segment; the tail's modes

**Housing** (Al 6082-T6, 3 mm wall): the water pressure at 8 m × 1.5 (0.12 MPa) on the outer tube and the front cap,
the rear cap's face held (its mount). By hand: the hoop stress `p r / t` and the elastic buckling of the tube under
external pressure (a long tube: `p_cr = E / (4 (1 − ν²)) (t / r)³`; between the caps it is stiffer — conservative).
**Leg** (PA12-CF, 30 % carbon, an input E 7 GPa, yield 70 MPa): the hip boss held, at the foot the pressing tripod's
load × 2 (a step) down and the current's share sideways. **Tail segment** (PA12-CF): the knuckle held, at its far end
the full bollard thrust across it (the tail bent 90° — the worst case for segment 1). Then the tail's modes: segment
1 with the rest of the tail lumped at its end (segment 2, shroud, motor) against the shaft and blade-pass orders, and
the servo's own mode (kp on the tail's inertia).

In [ ]:
AL6082 = talos.Material("Al 6082-T6 tube", youngs_modulus=70000.0, poissons_ratio=0.33, density=2.7e-9, yield_strength=260.0, source="EN 755-2")
PA12CF = talos.Material("PA12-CF (30 % carbon, printed)", youngs_modulus=7000.0, poissons_ratio=0.35, density=1.15e-9, yield_strength=70.0, source="assumed: datasheet class, printed in-plane")
depth, SF_p = 8.0, 1.5
p_ext = RHO * G * depth * SF_p * 1e-6                                # MPa
r_o, t_w, L_h = p["housing_diameter"] / 2, p["housing_wall"], p["housing_length"]
info = talos.inspect_step(cad["housing"].artifacts["step"], "mm-N-MPa")
outer = [s.tag for s in info.surfaces if s.kind != "Plane" and s.bbox_max[1] > r_o - 0.2]
HOUSING_REGIONS = [talos.Surfaces("outer_tube", outer), talos.SurfacesOnPlane("front", "x", L_h / 2), talos.SurfacesOnPlane("rear", "x", -L_h / 2)]
hmod = talos.StructuralModel(cad["housing"].artifacts["step"], "mm-N-MPa", AL6082, HOUSING_REGIONS, [talos.FixedSupport("rear")],
                             [talos.Pressure("outer_tube", p_ext), talos.Pressure("front", p_ext)], talos.MeshSettings(element_size=4.0, min_element_size=1.5), name="housing_8m")
hmod.mesh(RUNS / "housing_8m"); housing_res = hmod.solve(RUNS / "housing_8m")
r_m = r_o - t_w / 2
sigma_hoop = p_ext * r_m / t_w
p_cr = AL6082.youngs_modulus / (4 * (1 - AL6082.poissons_ratio ** 2)) * (t_w / r_m) ** 3
leg_F = (W_wet + T_press) / 3 * 2.0; leg_side = F_cur / 3
fx_, fy_, fz_ = lobster.SikarianLobster.foot_offset(p); rf = p["foot_diameter"] / 2 + 0.5; rb = p["leg_width"] * 0.8 + 0.5
LEG_REGIONS = [talos.SurfacesInBox("hip", (-rb, -rb, -rb, rb, rb, rb)), talos.SurfacesInBox("foot", (fx_ - rf, fy_ - rf, fz_ - rf, fx_ + rf, fy_ + rf, fz_ + rf))]
lmod = talos.StructuralModel(cad["leg"].artifacts["step"], "mm-N-MPa", PA12CF, LEG_REGIONS, [talos.FixedSupport("hip")],
                             [talos.Force("foot", fy=leg_side, fz=leg_F)], talos.MeshSettings(element_size=3.0, min_element_size=1.0), name="leg_press")
lmod.mesh(RUNS / "leg_press"); leg_res = lmod.solve(RUNS / "leg_press")
L1, dt_ = p["tail_segment_1"], p["tail_diameter"]
TAIL_REGIONS = [talos.SurfacesInBox("knuckle", (-dt_ * 0.45, -dt_ * 0.45, -dt_ * 0.45, dt_ * 0.45, dt_ * 0.45, dt_ * 0.45)), talos.SurfacesOnPlane("end", "x", -L1)]
tmod = talos.StructuralModel(cad["tail_segment"].artifacts["step"], "mm-N-MPa", PA12CF, TAIL_REGIONS, [talos.FixedSupport("knuckle")],
                             [talos.Force("end", fz=Tmax)], talos.MeshSettings(element_size=4.0, min_element_size=1.5), name="tail_thrust")
tmod.mesh(RUNS / "tail_thrust"); tail_res_fea = tmod.solve(RUNS / "tail_thrust")
def row(res, load):
    mt = res.metrics
    return {"load": load, "max von Mises [MPa]": mt.get("max_von_mises"), "SF yield": mt.get("safety_factor_yield"), "max displacement [mm]": mt.get("max_displacement")}
fea = pd.DataFrame({f"housing at {depth:g} m × {SF_p}": row(housing_res, f"{p_ext:.3f} MPa"),
                    "housing, hoop stress by hand": {"load": f"{p_ext:.3f} MPa", "max von Mises [MPa]": sigma_hoop, "SF yield": AL6082.yield_strength / sigma_hoop, "max displacement [mm]": float("nan")},
                    "housing, buckling by hand (long tube)": {"load": f"p_cr {p_cr:.2f} MPa", "max von Mises [MPa]": float("nan"), "SF yield": p_cr / p_ext, "max displacement [mm]": float("nan")},
                    "leg: pressing tripod × 2 + current": row(leg_res, f"{leg_F:.1f} N + {leg_side:.1f} N"),
                    f"tail segment 1: {Tmax:.0f} N across its end": row(tail_res_fea, f"{Tmax:.0f} N")}).T
print(f"buckling of the housing tube at {p_cr:.2f} MPa = {p_cr / (RHO * G) * 1e6:.0f} m of water: the 3 mm wall is set by machining and the O-ring grooves, not by the depth")
fea.round(3)

In [ ]:
tviz.show(tviz.plot_results(housing_res, field="von_mises"))

In [ ]:
m_out = (b["parts"]["shroud + propeller (PA12, CAD)"][0] + lr.THRUSTER["motor"].mass_kg + 2 * act.get(lr.SERVO_KEYS["tail"]).mass_g / 1000
         + cad_num["tail_segment_volume"] * 1e-9 * lr.MATERIALS["PA12-CF legs and arms"] * p["tail_segment_2"] / p["tail_segment_1"])
m_added = RHO * math.pi * (p["shroud_inner"] / 2000 + p["shroud_wall"] / 1000) ** 2 * p["shroud_length"] / 1000      # the water the shroud carries (added mass ~ its envelope)
tm = talos.StructuralModel(cad["tail_segment"].artifacts["step"], "mm-N-MPa", PA12CF, TAIL_REGIONS, [talos.FixedSupport("knuckle")], [],
                           talos.MeshSettings(element_size=5.0, min_element_size=2.0), name="tail_modal", masses=[talos.PointMass("end", (m_out + m_added) * 1e-3)])
tm.mesh(RUNS / "tail_modal"); tail_modes = tm.solve_modes(RUNS / "tail_modal", n_modes=4)
f_tail = tail_modes.metrics["frequencies_hz"]
kp_tail = lr.GAINS["tail"][0]
lever = (p["tail_segment_1"] + p["tail_segment_2"] + p["shroud_length"] / 2) / 1000
I_tail = (m_out + m_added) * lever ** 2 + lr.GAINS["tail"][2]
f_servo = math.sqrt(kp_tail / I_tail) / (2 * math.pi)
rpm_op = np.array([2400, 2600, 3300])
orders = pd.DataFrame({"shaft [Hz]": rpm_op / 60, "blade pass [Hz]": 3 * rpm_op / 60}, index=[f"{r} rpm" for r in rpm_op])
print(f"tail segment with {m_out + m_added:.2f} kg at its end (incl. {m_added:.2f} kg of water in the shroud): {[round(f, 0) for f in f_tail]} Hz | "
      f"the tail servo's own mode (kp {kp_tail} N·m/rad on {I_tail:.4f} kg·m²): {f_servo:.1f} Hz")
fig, ax = plt.subplots(figsize=(9, 3.6))
rr = np.linspace(0, 3500, 50)
ax.plot(rr, rr / 60, label="shaft (1×)"); ax.plot(rr, 3 * rr / 60, label="blade pass (3×)")
for f in f_tail[:3]:
    ax.axhline(f, color="r", ls="--", lw=0.8)
ax.axhline(f_servo, color="k", ls=":", lw=0.8, label="tail servo mode")
ax.set(xlabel="rpm", ylabel="Hz", title="Campbell diagram: the tail segment's modes (dashed) vs the propeller's orders", ylim=(0, max(250, f_tail[0] * 1.2))); ax.legend(fontsize=8); ax.grid(alpha=0.3)
orders.round(1)

## 8. Endurance

The 4S3P Li-ion pack (207 Wh, 85 % usable) against four states (the powers are inputs except the thruster's, which
is the motor model's at the swim rpm, and the walking servos', which come from the §2 walk: each servo draws its
stall current in proportion to its torque):

* **standby** — belly down on the bottom, the servos off, the computer asleep, the hydrophones listening: 3.5 W;
* **watch** — the same with the cameras and the imaging sonar on: 11 W;
* **walk** — standing up and walking (the twelve leg servos) + watch;
* **swim** — the thruster at 2600 rpm + watch.

The datasheet's 12–48 h is the range between watching and standby; a 24 h deployment (walk 20 min in, swim 5 min,
the rest on standby with the cameras and sonar on 10 % of the time) is the mission line.

In [ ]:
E = lr.THRUSTER["battery"].usable_wh
P_standby, P_watch = 3.5, 11.0                                          # inputs [W]
tau_w = np.abs(np.asarray(ep_walk.log["tau"])); tw_sel = (np.asarray(ep_walk.log["t"]) > t0 + 1.2) & (np.asarray(ep_walk.log["t"]) < t1)
P_legs = sum((tau_w[tw_sel, names.index(f"{leg}_{j}")] / LEG.stall_Nm * LEG.stall_A * LEG.voltage_V).mean() for leg in lr.LEGS for j in ("yaw", "lift"))
Q_sw = lr.thrust(tab, 2600, 0.8)[1]
V_sw, I_sw = mot.voltage_for(2600, Q_sw); P_thr = V_sw * I_sw
states = pd.DataFrame({"power [W]": {"standby": P_standby, "watch": P_watch, "walk": P_watch + P_legs, "swim": P_watch + P_thr}})
states["hours on the pack"] = E / states["power [W]"]
profile = {"walk": 20 / 60, "swim": 5 / 60}
rest = 24 - sum(profile.values())
watch_duty = 0.10                                                     # input
E24 = profile["walk"] * states.loc["walk", "power [W]"] + profile["swim"] * states.loc["swim", "power [W]"] + rest * ((1 - watch_duty) * P_standby + watch_duty * P_watch)
print(f"pack {lr.THRUSTER['battery'].energy_wh:.0f} Wh, {E:.0f} Wh usable | a 24 h deployment (20 min walking, 5 min swimming, the rest standby, watching {watch_duty:.0%} of it) "
      f"uses {E24:.0f} Wh ({E24 / E * 100:.0f} % of the pack)")
states.round(1)

## 9. The jobs in MuJoCo (ChironLab, underwater)

The Lobster as a Chiron robot (`designs/lobster_robot.py`): every body with MuJoCo's ellipsoid fluid shape (drag and
added mass in water — Chiron's new `density`/`viscosity` options), the 26 servos on the catalogue's torque-speed
lines, and the **`Water` hook**: buoyancy on every body at its centre of buoyancy (MuJoCo applies none) and the
thruster — thrust along the shroud axis from the BEMT table × the duct gain at the commanded rpm and the inflow, the
propeller's reaction torque. The scene (`designs/lobster_scenario.py`): a rough river bed; for the cut, the Ø600
concrete outfall bedded in it with 150 mm of silt inside, the Ø10 PP rope across its mouth from two stakes (two
hinged halves welded at the knot: the Manus's `WireCutter` hook parts them when both jaws squeeze 600 N for 50 ms; the
cut halves spring back to the stakes). The missions (`designs/lobster_controller.py`):

1. **swim and land** — released 1 m over the bed, legs tucked: the tail aims the thrust through the CG, a depth loop
   sets its angle, the yaw joints hold the heading; eases off, lowers the legs, lands on them;
2. **cut and enter** — walks to the rope, raises the right pincer into a fixed cutting pose with the jaws open,
   **creeps** the body forward until the rope is in the notch's throat, closes the worm drive (the contact forces
   decide when it parts), lowers the claw so the ends drop, backs off, stows and walks 1 m into the pipe;
3. **hold in a current** — 0.5 m/s across it (MuJoCo's `wind`): standing, it slides; it curls the tail over the
   back and presses itself down (the jet up, the line between the feet), and walks across the current pressing.

Each writes a movie (`output/24_sikarian_lobster_nefri/`; also `scenarios/sikarian_lobster.py`).

In [ ]:
def mission(kind):
    scene = ls.Scene()
    extra = dict(wind=(0.0, 0.5, 0.0), course_extent=(-2.0, 6.0, -2.0, 6.0)) if kind == "current" else {}
    lab = ls.make_lab(scene, kind, robot=robot, log_geoms=True, **extra)
    ep = ls.run(kind, scene, lab=lab)
    ep.save(RUNS / f"mission_{kind}.npz")
    return ep, ls.timeseries(ep)
def movie_of(ep, name, camera="follow"):
    dt = float(ep.log["t"][1] - ep.log["t"][0]); every = max(1, int(round(1 / (25 * dt))))
    imgs = cviz.frames(ep, camera=camera, every=every, size=(960, 540))
    return cviz.to_video(imgs, OUT / f"{name}.mp4", fps=25), imgs, every * dt
def starts(ep):
    out = {}
    for t0_, name, note in ep.log["mission"]:
        if note == "start": out.setdefault(name, t0_)
    return out
ep_swim, ts_swim = mission("swim")
st_swim = starts(ep_swim)
sw = ts_swim[(ts_swim.t > st_swim["swim"] + 3) & (ts_swim.t < st_swim.get("ease off and sink", ts_swim.t.iloc[-1]))]
print(f"swim: finished {ep_swim.log['mission_finished']} | cruise {sw.vx.mean():.2f} m/s at {sw.z.mean():.2f} m (hull centre), pitch within ±{sw.tilt_deg.abs().max():.1f}° | "
      f"landed at x = {ts_swim.x.iloc[-1]:.2f} m, tilt {ts_swim.tilt_deg.iloc[-1]:.1f}°")
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(ts_swim.t, ts_swim.z, label="hull z [m]"); ax[0].plot(ts_swim.t, ts_swim.vx, label="vx [m/s]"); ax[0].plot(ts_swim.t, ts_swim.tilt_deg / 10, label="tilt / 10 [deg]")
ax[0].set(xlabel="time [s]", title="swim and land"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for j in lr.TAIL_JOINTS:
    ax[1].plot(ts_swim.t, ts_swim[j + "_deg"], label=j)
ax[1].set(xlabel="time [s]", ylabel="deg", title="tail joints"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
for a in ax:
    for t_ in st_swim.values(): a.axvline(t_, color="#ddd", lw=0.6, zorder=0)
fig.tight_layout()
mv_swim, _, _ = movie_of(ep_swim, "lobster_swim_and_land")
display(Video(str(mv_swim), embed=False, width=900))

In [ ]:
ep_cut, ts_cut = mission("cut_and_enter")
st_cut = starts(ep_cut)
hist = np.asarray(ep_cut.log["cutter_history"]).reshape(-1, 3)
print(f"cut and enter: finished {ep_cut.log['mission_finished']} | rope cut at {ep_cut.log['cut_at']} s | events {ep_cut.log.get('events', [])}")
print(f"the Lobster ends {ts_cut.x.iloc[-1] - ls.Scene().x_mouth:.2f} m inside the pipe, tilt {ts_cut.tilt_deg.iloc[-1]:.1f}°")
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(ts_cut.t, ts_cut.x, label="hull x [m]"); ax[0].axhline(ls.Scene().x_mouth, color="k", ls=":", lw=0.8, label="the pipe's mouth")
ax[0].plot(ts_cut.t, ts_cut.tilt_deg / 10, label="tilt / 10 [deg]"); ax[0].set(xlabel="time [s]", title="cut and enter"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
if len(hist):
    ax[1].plot(hist[:, 0], hist[:, 1], label="upper jaw on the rope [N]"); ax[1].plot(hist[:, 0], hist[:, 2], label="lower jaw [N]")
ax[1].axhline(ls.F_CUT, color="r", ls="--", lw=0.8, label="F_CUT"); ax[1].set(xlabel="time [s]", title="the squeeze (WireCutter)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
for a in ax:
    for t_ in st_cut.values(): a.axvline(t_, color="#ddd", lw=0.6, zorder=0)
fig.tight_layout()
mv_cut, imgs_cut, dt_cut = movie_of(ep_cut, "lobster_cut_and_enter")
fig, axes = plt.subplots(2, 3, figsize=(18, 6.5))
for a, (name, off) in zip(axes.flat, (("walk to the rope", 5.0), ("R claw: cutting pose, jaws open", 3.5), ("cut", 0.5),
                                      ("R claw: down, the ends drop", 2.5), ("walk into the pipe", 4.0), ("walk into the pipe", 14.0))):
    j = min(len(imgs_cut) - 1, int(round((st_cut.get(name, 0.0) + off) / dt_cut)))
    a.imshow(imgs_cut[j]); a.set_axis_off(); a.set_title(f"{name} (t = {j * dt_cut:.1f} s)", fontsize=9)
fig.tight_layout()
display(Video(str(mv_cut), embed=False, width=900))

In [ ]:
ep_cur, ts_cur = mission("current")
st_cur = starts(ep_cur)
def drift(a, b_):
    s = ts_cur[(ts_cur.t > a) & (ts_cur.t < b_)]
    return (s.y.iloc[-1] - s.y.iloc[0]) / (s.t.iloc[-1] - s.t.iloc[0])
v_free = drift(0.5, st_cur["curl the tail over the back"])
v_held = drift(st_cur["press down with the thrust"] + 0.5, st_cur["walk across the current, pressing"])
v_walk_c = drift(st_cur["walk across the current, pressing"] + 1.0, st_cur["stand, pressing"])
print(f"hold in a 0.5 m/s current: finished {ep_cur.log['mission_finished']} | drift standing {v_free:.3f} m/s, pressing {v_held:.3f} m/s, "
      f"walking across pressing {v_walk_c:.3f} m/s (downstream +) | max tilt {ts_cur.tilt_deg.max():.1f}°")
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(ts_cur.t, ts_cur.y, label="y (downstream) [m]"); ax.plot(ts_cur.t, ts_cur.x, label="x [m]"); ax.plot(ts_cur.t, ts_cur.foot_load_N / 10, label="feet / 10 [N]")
for t_ in st_cur.values(): ax.axvline(t_, color="#ddd", lw=0.6, zorder=0)
ax.set(xlabel="time [s]", title="in the current: slide, press, walk across"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
mv_cur, _, _ = movie_of(ep_cur, "lobster_hold_in_current")
display(Video(str(mv_cur), embed=False, width=900))

## 10. Export and next steps

In [ ]:
doc = {"source": "24_sikarian_lobster_nefri", "design": {"file": "designs/lobster.py", "parameters": p}, "job": JOB.to_dict(),
       "mass_kg": b["mass"], "volume_L": b["volume"] * 1000, "wet_weight_N": b["wet_weight_N"], "foam_L": b["foam_volume"] * 1000, "trim": robot.trim,
       "budget": {k: {"kg": v[0], "L": v[1] * 1000} for k, v in b["parts"].items()}, "legs": legs.to_dict(orient="index"), "walk_m_s": v_walk,
       "bollard": bollard.to_dict(orient="index"), "swim_speed_model_m_s": v_swim, "cavitation": cav.to_dict(orient="index"),
       "tail_envelope": envelope.to_dict(orient="index"), "cfd_prop": compare_prop.to_dict(orient="index"), "cfd_jet": jet.to_dict(orient="index"),
       "ropes": ropes.to_dict(orient="index"), "jaw_fea": jaw_res.metrics, "fea": fea.to_dict(orient="index"), "tail_modes_hz": f_tail, "tail_servo_mode_hz": f_servo,
       "endurance": states.to_dict(orient="index"), "deployment_24h_Wh": E24,
       "missions": {"swim": {"finished": ep_swim.log["mission_finished"], "movie": str(mv_swim)},
                    "cut_and_enter": {"finished": ep_cut.log["mission_finished"], "cut_at": ep_cut.log["cut_at"], "movie": str(mv_cut)},
                    "current": {"finished": ep_cur.log["mission_finished"], "drift_standing": v_free, "drift_pressing": v_held, "movie": str(mv_cur)}}}
(RUNS / "sikarian_lobster_nefri.json").write_text(json.dumps(doc, indent=2, default=float))
print("written:", RUNS / "sikarian_lobster_nefri.json")

**What the numbers say** — read them above; the ones that set the design: the foam and the trim (§1: a 10 % net weight
is ~7 N — the legs need little, a current needs more, the thrust supplies it), the pressing case for the leg servos
(§2), the duct and the motor at bollard on the 4S pack (§3), the tail's narrow through-the-CG envelope and its joint-1
torque at full thrust (§4 — the controller limits the rpm when the tail is bent, or the tail servos go up a size), the
cutting margin of the worm drive on the rope classes (§6: Nefri cuts PP and nylon; an HMPE line or a power cable is
Ornatus' job), the housing far from buckling at 8 m (§7), and in §9 whether each job finished.

**Next steps an engineer would take:** a rope-cutter test on the real rope and the real notch (F_CUT is the one
input the cut stands on); the duct's gain measured on a bollard rig (and a CFD case with a stationary duct, e.g. a
sliding mesh); perception for the rope (the mission knows where it is); a tail-joint rpm limit in the controller;
sea water (2.5 % more buoyancy: the foam goes down by ~0.15 L) and fouling; the cable tether or acoustic modem for the
watch data; and **Ornatus** — 15 kg, a Ø12 mm electric cable (≈2.5 kN to cut: a bigger worm drive and a shear blade),
the same Ø600 pipe (its width is the limit: the feet span has to stay inside the 0.52 m silt floor) — notebook 25.

**Other uses of the same machine** (a note, not modelled here):

* **Infrastructure inspection** — walking the inside of outfalls, culverts and intake pipes and along bridge piers,
  quay walls and dam faces at the bottom, where a swimming ROV is pushed around by the current: the legs hold it on
  the structure, the cameras and the imaging sonar look at the wall from a fixed distance, the pincers clear debris
  and hold it against the current while it looks (the "hold in a current" job of §9 is the core of it). What
  changes: lights, a laser scaler for crack sizing, a tether or an acoustic modem for the data, and the pipe's own
  diameter range (Ø600 is the lower limit for Nefri's feet).
* **Environmental sampling** — sediment, water and biofilm samples at set points on the bottom of rivers, lakes and
  harbours, and long standby stations with sensors (temperature, conductivity, turbidity, dissolved oxygen):
  it walks to the point without stirring the silt the way a thruster does, the pincer takes a grab or a core, and it
  sits on standby for days (§8: ~50 h on the pack). What changes: a sample carousel in place of part of the foam
  (re-trim, §1), a sensor head on the mast, and a low-power wake-up schedule.